In [1]:
import pandas as pd
from os import listdir
from zipfile import ZipFile

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

def morbidity(data_dir):
    data_dir = Path(data_dir)

    def _read_csv(name):
        return pd.read_csv(data_dir / name, encoding="utf-8-sig")

    def _calc(yld_df, yll_df, scen):

        year_cols = sorted(
            [c for c in yld_df.columns if str(c).isdigit() and c in yll_df.columns],
            key=int
        )
        
        key_cols = [c for c in ["cause_name", "ISO3", "sex_name", "age_name"]
                    if c in yld_df.columns and c in yll_df.columns]

        yld = yld_df.set_index(key_cols)[year_cols]
        yll = yll_df.set_index(key_cols)[year_cols]

        # 같은 key + 같은 year끼리 element-wise 나눗셈
        ratio = yld.divide(yll).replace([np.inf, -np.inf], np.nan).fillna(0)

        out = ratio.reset_index()
        out["measure_name"] = "Morbidity (YLD/YLL)"
        out = out[key_cols + ["measure_name"] + year_cols]
        out.to_csv(data_dir / f"morbidity_{scen}.csv", index=False, encoding="utf-8-sig")
        return out

    yld_lower = _read_csv("YLDs_lower.csv")
    yld_upper = _read_csv("YLDs_upper.csv")
    yld_val   = _read_csv("YLDs_val.csv")
    yll_lower = _read_csv("YLLs_lower.csv")
    yll_upper = _read_csv("YLLs_upper.csv")
    yll_val   = _read_csv("YLLs_val.csv")

    return {
        "lower": _calc(yld_lower, yll_lower, "lower"),
        "upper": _calc(yld_upper, yll_upper, "upper"),
        "val":   _calc(yld_val,   yll_val,   "val"),
    }

In [9]:
IHD = morbidity("../../data/ASCVD/IHD/")
IS = morbidity("../../data/ASCVD/IS/")
PAD = morbidity("../../data/ASCVD/PAD/")